# Spark ETL — flights from Kafka to results

The **core processing stage**: `Kafka (group10.flights) -> Spark -> clean -> filter commercial -> join reference data -> aggregate -> parquet`.

Computes the analysis questions and stores each result as parquet:
- **Q1 Dominance** — top airlines / carrier nations
- **Q2 Geography** — flight-density grid (corridors)
- **Q3 Busiest airports** — spatial join (nearest airport)
- **Q4a Departures vs arrivals** — split by climb/descent
- **Extras** — altitude histogram + heading flow-field (for non-bar visuals)

**Run on the JupyterHub.** Reads **live** from Kafka, so make sure the producer has run recently.

## 1. SparkSession
Connect to the cluster master and pull in the Kafka connector (matched to Spark 3.5.6). First run on a fresh kernel downloads the jars via Ivy (one-time).

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("group10-flights-etl")
    .master("spark://172.29.16.102:7077")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.6")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| app", spark.sparkContext.applicationId)

## 2. Read from Kafka and parse the JSON
Batch-read the whole topic and parse JSON into typed columns. `.cache()` pins ONE consistent read (the live topic keeps growing, so without it every action would re-read different data).

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, BooleanType, LongType,
)

schema = StructType([
    StructField("icao24", StringType()),
    StructField("callsign", StringType()),
    StructField("origin_country", StringType()),
    StructField("longitude", DoubleType()),
    StructField("latitude", DoubleType()),
    StructField("baro_altitude", DoubleType()),
    StructField("on_ground", BooleanType()),
    StructField("velocity", DoubleType()),
    StructField("true_track", DoubleType()),
    StructField("vertical_rate", DoubleType()),
    StructField("geo_altitude", DoubleType()),
    StructField("snapshot_time", LongType()),
])

raw = (spark.read.format("kafka")
    .option("kafka.bootstrap.servers", "172.29.16.101:9092")
    .option("subscribe", "group10.flights")
    .option("startingOffsets", "earliest")
    .load())

flights = raw.select(F.from_json(F.col("value").cast("string"), schema).alias("f")).select("f.*").cache()
print("parsed flights:", flights.count())

## 3. Clean
Keep airborne aircraft with a callsign; derive `airline_icao` (first 3 chars of the callsign).

In [ ]:
clean = (flights
    .filter(~F.col("on_ground"))
    .filter(F.col("callsign") != "")
    .withColumn("airline_icao", F.substring("callsign", 1, 3)))
print("airborne w/ callsign:", clean.count())

## 4. Keep only commercial flights (broadcast join with the airline reference)
Load the scraped airline-codes lookup from GitHub, broadcast it, and **inner-join** on the ICAO code — this is the commercial filter (private/military/GA drop out). Cached because every question uses it.

In [ ]:
import pandas as pd

AIRLINES_CSV = "https://raw.githubusercontent.com/jakobiene/bde-project-group10/main/data/processed/airlines.csv"
airlines = spark.createDataFrame(pd.read_csv(AIRLINES_CSV))

commercial = clean.join(F.broadcast(airlines), clean.airline_icao == airlines.icao, "inner").cache()
print("commercial flights:", commercial.count())

## 5. Q1 — Dominance
Distinct aircraft per airline and per operator country, stored to parquet.

In [ ]:
by_airline = (commercial.groupBy("airline_icao", "airline", "country")
    .agg(F.countDistinct("icao24").alias("aircraft"))
    .orderBy(F.desc("aircraft")))

by_country = (commercial.groupBy("origin_country")
    .agg(F.countDistinct("icao24").alias("aircraft"))
    .orderBy(F.desc("aircraft")))

by_airline.show(10, truncate=False)
by_country.show(10, truncate=False)

In [ ]:
import os
# notebook runs from notebooks/, so go up one level into the repo-root data/processed
OUT_DIR = "../data/processed"
os.makedirs(OUT_DIR, exist_ok=True)
by_airline.toPandas().to_parquet(f"{OUT_DIR}/dominance_by_airline.parquet", index=False)
by_country.toPandas().to_parquet(f"{OUT_DIR}/dominance_by_country.parquet", index=False)
print("Q1 results written to", os.path.abspath(OUT_DIR))

## 6. Q2 — Geography: flight-density grid (corridors)
Round positions to a ~0.1° grid and count records per cell → feeds the density map (heatmap) in notebook 05.

In [ ]:
density = (commercial
    .withColumn("lat_bin", F.round("latitude", 1))
    .withColumn("lon_bin", F.round("longitude", 1))
    .groupBy("lat_bin", "lon_bin")
    .agg(F.count("*").alias("flights"))
    .orderBy(F.desc("flights")))

density.show(10)
density.toPandas().to_parquet(f"{OUT_DIR}/density_grid.parquet", index=False)
print("Q2 density grid written")

## 7. Q3 — Busiest airports (spatial join)
Pick arrival/departure candidates (low + climbing/descending), then match each to its nearest airport via haversine distance (within 30 km).

In [ ]:
AIRPORTS_CSV = "https://raw.githubusercontent.com/jakobiene/bde-project-group10/main/data/processed/airports.csv"
airports = spark.createDataFrame(pd.read_csv(AIRPORTS_CSV))
print("airports:", airports.count())

arrdep = (commercial
    .filter(F.col("geo_altitude").isNotNull())
    .filter((F.col("geo_altitude") >= 0) & (F.col("geo_altitude") < 3000))
    .filter(F.abs(F.col("vertical_rate")) > 1)
    .select("icao24", "callsign", "latitude", "longitude", "vertical_rate", "geo_altitude"))
print("arrival/departure candidates:", arrdep.count())

Cross-join candidates with airports, haversine distance, keep the nearest within 30 km. `nearest` is cached (Q3 and Q4a both use it).

In [ ]:
from pyspark.sql.window import Window
R = 6371.0  # earth radius, km

ap = airports.select(
    F.col("icao_code"), F.col("name").alias("airport_name"),
    F.col("iso_country"), F.col("latitude_deg"), F.col("longitude_deg"))

pairs = arrdep.crossJoin(F.broadcast(ap)).withColumn(
    "dist_km",
    2 * R * F.asin(F.sqrt(
        F.pow(F.sin(F.radians(F.col("latitude_deg") - F.col("latitude")) / 2), 2)
        + F.cos(F.radians(F.col("latitude"))) * F.cos(F.radians(F.col("latitude_deg")))
        * F.pow(F.sin(F.radians(F.col("longitude_deg") - F.col("longitude")) / 2), 2)
    )))

w = Window.partitionBy("icao24").orderBy("dist_km")
nearest = (pairs.withColumn("rk", F.row_number().over(w))
    .filter(F.col("rk") == 1)
    .filter(F.col("dist_km") <= 30)).cache()

busiest = (nearest.groupBy("icao_code", "airport_name", "iso_country")
    .agg(F.countDistinct("icao24").alias("aircraft"))
    .orderBy(F.desc("aircraft")))
busiest.show(15, truncate=False)
busiest.toPandas().to_parquet(f"{OUT_DIR}/busiest_airports.parquet", index=False)
print("Q3 results written")

## 8. Q4a — Departures vs arrivals per airport
Split the nearest-airport matches by phase: climbing = just departed, descending = arriving.

In [ ]:
labeled = nearest.withColumn(
    "phase", F.when(F.col("vertical_rate") > 1, "departure").otherwise("arrival"))

top_departures = (labeled.filter(F.col("phase") == "departure")
    .groupBy("icao_code", "airport_name", "iso_country")
    .agg(F.countDistinct("icao24").alias("departures"))
    .orderBy(F.desc("departures")))

top_arrivals = (labeled.filter(F.col("phase") == "arrival")
    .groupBy("icao_code", "airport_name", "iso_country")
    .agg(F.countDistinct("icao24").alias("arrivals"))
    .orderBy(F.desc("arrivals")))

print("Top departure airports:")
top_departures.show(10, truncate=False)
print("Top arrival airports:")
top_arrivals.show(10, truncate=False)

top_departures.toPandas().to_parquet(f"{OUT_DIR}/top_departures.parquet", index=False)
top_arrivals.toPandas().to_parquet(f"{OUT_DIR}/top_arrivals.parquet", index=False)
print("Q4a results written")

## 9. Extras — altitude histogram & heading flow-field
Two more small aggregates that power the non-bar visuals in notebook 05:
- **altitude histogram:** flight records per 500 m band → reveals the discrete flight levels.
- **flow field:** per ~1° cell, the mean heading (mean of sin/cos of `true_track`) + count → an arrow/flow map of how traffic moves.

In [ ]:
# Altitude histogram: flight records per 500 m band (outliers dropped)
alt_hist = (commercial
    .filter(F.col("geo_altitude").isNotNull()
            & (F.col("geo_altitude") >= 0) & (F.col("geo_altitude") < 14000))
    .withColumn("alt_bin", (F.floor(F.col("geo_altitude") / 500) * 500).cast("int"))
    .groupBy("alt_bin").agg(F.count("*").alias("flights"))
    .orderBy("alt_bin"))
alt_hist.toPandas().to_parquet(f"{OUT_DIR}/altitude_hist.parquet", index=False)

# Flow field: per ~1deg cell, mean heading via mean(sin), mean(cos) of true_track + count
flow = (commercial
    .filter(F.col("true_track").isNotNull())
    .withColumn("lat_c", F.round("latitude", 0))
    .withColumn("lon_c", F.round("longitude", 0))
    .withColumn("trk", F.radians("true_track"))
    .groupBy("lat_c", "lon_c")
    .agg(F.count("*").alias("flights"),
         F.avg(F.sin("trk")).alias("mean_u"),
         F.avg(F.cos("trk")).alias("mean_v")))
flow.toPandas().to_parquet(f"{OUT_DIR}/flow_field.parquet", index=False)

print("altitude_hist bins:", alt_hist.count(), "| flow_field cells:", flow.count())

## Summary
Kafka → Spark → storage ETL. Produced and stored to `data/processed/`:
- `dominance_by_airline`, `dominance_by_country` (Q1)
- `density_grid` (Q2)
- `busiest_airports` (Q3)
- `top_departures`, `top_arrivals` (Q4a)
- `altitude_hist`, `flow_field` (extra visuals)

Notebook 05 reads these parquet files and builds the charts / maps.